# Aspen HYSYS Interface Tutorial

This notebook is a teaching-oriented walkthrough of the Python-to-HYSYS COM
interface used in the HDA workflow.

The structure is intentionally explicit:

1. choose a runtime mode
2. connect to HYSYS or fall back to a mock case
3. inspect the case map used by the helper functions
4. read a few streams before changing anything
5. apply a structured sample of flowsheet updates
6. compare baseline and tuned cases
7. inspect energy-stream usage after the run


## Before You Start

Use `mock` mode if you are reading this on GitHub or do not have Aspen HYSYS.

Switch to `active_hysys` mode only when all of the following are true:

- you are on a Windows machine
- Aspen HYSYS is already open
- the target case is the active document in HYSYS
- `pywin32` is installed in the Python environment

The notebook defaults to mock mode so every section can still run end to end.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
DEMO_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from hysys_demo import (
    DEFAULT_HDA_SAMPLE,
    HdaFlowsheetMap,
    UTILITY_COST_USD_PER_KJ,
    apply_hda_demo_sample,
    build_mock_context,
    calculate_utility_cost_per_hour,
    collect_energy_table,
    connect_to_active_case,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)


## 1. Select the Runtime Mode

The helper package supports two modes:

- `mock`: uses a fake in-memory HDA case that mimics the parts of the COM
  interface used by the tutorial
- `active_hysys`: calls `win32com.client.GetObject(None, "HYSYS.Application")`
  and reads the active HYSYS document


In [ ]:
RUNTIME_MODE = 'mock'  # change to 'active_hysys' on Windows with Aspen HYSYS open

if RUNTIME_MODE == 'active_hysys':
    ctx = connect_to_active_case()
elif RUNTIME_MODE == 'mock':
    ctx = build_mock_context()
else:
    raise ValueError("RUNTIME_MODE must be 'mock' or 'active_hysys'")

print('Runtime mode:', RUNTIME_MODE)
print('Case title:', ctx.case.Title)


## 2. Understand the Case Map

The automation helpers should not hard-code stream and unit names all over
this notebook. Instead, they use one mapping object that tells the helper
package where the hydrogen feed, reactor, splitters, and columns live in the
HYSYS case.


In [ ]:
mapping = HdaFlowsheetMap()
mapping_df = pd.DataFrame(
    [{'field': key, 'hysys_name': value} for key, value in vars(mapping).items()]
)
mapping_df


## 3. Inspect a Few Streams Before Writing Anything

A good habit with HYSYS automation is to first read a small set of streams and
verify that the notebook is pointing at the right objects.


In [ ]:
def snapshot_streams(stream_names):
    rows = []
    for name in stream_names:
        stream = ctx.flowsheet.MaterialStreams.Item(name)
        rows.append(
            {
                'stream': name,
                'temperature_c': getattr(stream.Temperature, 'Value', None),
                'pressure_kpa': getattr(stream.Pressure, 'Value', None),
                'molar_flow_kmol_s': getattr(stream.MolarFlow, 'Value', None),
            }
        )
    return pd.DataFrame(rows)


inspection_df = snapshot_streams(
    [
        mapping.hydrogen_stream,
        mapping.feed_stream,
        mapping.reactor_outlet_stream,
        mapping.condenser_stream,
        mapping.benzene_column_stream,
        mapping.toluene_column_stream,
    ]
)
inspection_df


## 4. Review the Baseline Input Sample

The sample dictionary is the notebook-facing representation of the flowsheet
changes we want to apply. The helper package translates those high-level
fields into individual stream, splitter, reactor, and column updates.


In [ ]:
baseline_sample_df = pd.DataFrame(
    [{'parameter': key, 'value': value} for key, value in DEFAULT_HDA_SAMPLE.items()]
)
baseline_sample_df


## 5. Apply the Baseline Sample

`apply_hda_demo_sample()` performs the structured writes. In this demo it:

- sets the hydrogen-feed flow
- updates the reactor-feed temperature and pressure
- links the secondary feed-stream pressure when needed
- updates condenser and column pressures
- updates reactor volume
- updates purge and recycle splitter fractions
- updates column tray counts and feed-stage ratios


In [ ]:
baseline_applied = apply_hda_demo_sample(DEFAULT_HDA_SAMPLE, mapping=mapping, context=ctx)

baseline_snapshot_df = snapshot_streams(
    [
        mapping.hydrogen_stream,
        mapping.feed_stream,
        mapping.reactor_outlet_stream,
        mapping.condenser_stream,
    ]
)
baseline_snapshot_df


## 6. Create and Apply a Tuned Case

Once the baseline mapping works, the usual workflow is to clone the sample,
adjust a few values, and run again.


In [ ]:
tuned_sample = dict(DEFAULT_HDA_SAMPLE)
tuned_sample.update(
    {
        'operation_mode': 'isothermal',
        'feed_pressure_kpa': 3450.0,
        'feed_temperature_c': 640.0,
        'recycle_split_fraction': 0.86,
        'purge_split_fraction': 0.08,
        'benzene_column_trays': 48,
        'toluene_column_trays': 30,
    }
)

tuned_sample_df = pd.DataFrame(
    [{'parameter': key, 'value': value} for key, value in tuned_sample.items()]
)
tuned_sample_df


In [ ]:
def run_case(label, sample):
    applied = apply_hda_demo_sample(sample, mapping=mapping, context=ctx)
    energy_df = collect_energy_table(context=ctx)
    return {
        'label': label,
        'operation_mode': applied['operation_mode'],
        'feed_pressure_kpa': applied['feed_pressure_kpa'],
        'feed_temperature_c': applied['feed_temperature_c'],
        'recycle_split_fraction': applied['recycle_split_fraction'],
        'purge_split_fraction': applied['purge_split_fraction'],
        'benzene_column_trays': applied['benzene_column_trays'],
        'toluene_column_trays': applied['toluene_column_trays'],
        'energy_stream_count': len(energy_df),
        'utility_cost_usd_per_h': calculate_utility_cost_per_hour(context=ctx),
    }


comparison_df = pd.DataFrame(
    [
        run_case('baseline', DEFAULT_HDA_SAMPLE),
        run_case('tuned', tuned_sample),
    ]
)
comparison_df


## 7. Inspect the Energy Streams

After applying a case, the next useful check is usually the energy-stream
table. This confirms which duties are present and how the helper grouped them
into utility categories.


In [ ]:
energy_df = collect_energy_table(context=ctx)
energy_df


In [ ]:
utility_summary_df = (
    energy_df.groupby('utility_category', dropna=False)[['heat_flow_kw', 'heat_flow_kj_h']]
    .sum()
    .reset_index()
    .sort_values('utility_category')
)
utility_summary_df['estimated_cost_usd_per_h'] = utility_summary_df.apply(
    lambda row: row['heat_flow_kj_h'] * UTILITY_COST_USD_PER_KJ[row['utility_category']],
    axis=1,
)
utility_summary_df


## 8. How to Retarget This Notebook to Your Own HYSYS Case

The shortest path is:

1. duplicate `HdaFlowsheetMap` with your own stream and operation names
2. check each mapped name interactively with small read-only snapshots first
3. replace `DEFAULT_HDA_SAMPLE` with parameters that make sense for your case
4. extend `apply_hda_demo_sample()` only when your unit-operation logic is
   genuinely different

Common failure modes:

- wrong stream or operation names in the map
- trying to connect when no HYSYS case is active
- running outside Windows without `pywin32`
- mixing notebook logic and COM write logic instead of keeping the write path
  inside the helper package
